In [13]:
# Import 3 hal
import os
import numpy as np
import cv2

In [14]:
# inisialisasi seluruh PPATH
TRAIN_PATH = "./Dataset/train"
TEST_PATH = "./Dataset/test"
MODEL_PATH = "./patrick_orl_lbph_model.xml"

In [15]:
print(os.listdir(cv2.data.haarcascades))

['haarcascade_eye.xml', 'haarcascade_eye_tree_eyeglasses.xml', 'haarcascade_frontalcatface.xml', 'haarcascade_frontalcatface_extended.xml', 'haarcascade_frontalface_alt.xml', 'haarcascade_frontalface_alt2.xml', 'haarcascade_frontalface_alt_tree.xml', 'haarcascade_frontalface_default.xml', 'haarcascade_fullbody.xml', 'haarcascade_lefteye_2splits.xml', 'haarcascade_license_plate_rus_16stages.xml', 'haarcascade_lowerbody.xml', 'haarcascade_profileface.xml', 'haarcascade_righteye_2splits.xml', 'haarcascade_russian_plate_number.xml', 'haarcascade_smile.xml', 'haarcascade_upperbody.xml', '__init__.py', '__pycache__']


In [16]:
# inisialisasi face_cascades
face_cascades = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# **Bagian Awal**
isinya fungsi:
* preprocess_image(face_img)
* detect_and_crop_face(gray_img)
* load_data(folder_path, class_name=None)

In [17]:
def preprocess_image(face_img):
    face_img = cv2.resize(face_img, (100,100))
    face_img = cv2.equalizeHist(face_img)
    return face_img

In [18]:
def detect_and_crop_face(gray_img):
    # dapatkan detected_face
    detected_face = face_cascades.detectMultiScale(
        gray_img,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30,30)
    )

    # validasi detected_face
    if len(detected_face) < 1:
        return None, None
    
    # dapatkan face_img dan face_box
    x, y, w, h = max(detected_face, key=lambda rect: rect[2]*rect[3])
    face_img = gray_img[y:y+h, x:x+w] #[baris, kolom]
    face_img = preprocess_image(face_img)
    face_box = (x, y, w, h)
    return face_img, face_box

In [19]:
def load_data(folder_path, class_name=None):
    print(f"Loading: {folder_path}")
    # inisialisasi face_list dan label_list
    face_list = []
    label_list = []

    # validasi class_name
    if class_name is None:
        class_name = sorted(os.listdir(folder_path))
    
    # for loop untuk setiap kelas di folder tsb
    for label, person_name in enumerate(class_name):
        person_folder = os.path.join(folder_path, person_name)

        # validasi jika person_folder adalah directory
        if not os.path.isdir(person_folder):
            continue

        #for loop untuk setiap gambar di person_folder
        for img_name in sorted(os.listdir(person_folder)):
            #validasi jika formatnya sesuai
            if not img_name.lower().endswith((".pgm", ".jpg", ".png")):
                continue

            # dapatkan gray_img dari img_path, validasi gray_img
            img_path = os.path.join(person_folder, img_name)
            gray_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if gray_img is None:
                continue

            # dapatkan face_img saja dan validasikan
            face_img, _ = detect_and_crop_face(gray_img)
            if face_img is None:
                continue

            # masukkan face_img dan label ke list masing"
            face_list.append(face_img)
            label_list.append(label)
    
    # return face_list, label_list yang sudah jadi array int32, class_name
    return face_list, np.array(label_list, dtype=np.int32), class_name

# **Bagian Kedua**
isinya fungsi:
* train_test_model()
* load_saved_model()
* predict_image()

In [20]:
def train_test_model():
    print("Training Model...")
    if os.path.exists(MODEL_PATH):
        print("No need for training because there is already a model!")
        return

    # Train: inisialisasi class_name, face_recognizer, dan seluruh data training (face, label, class_name)
    train_faces, train_labels, class_name = load_data(TRAIN_PATH)
    if len(train_faces) < 1:
        print("WARNING: can not read train data")
        return

    face_recognizer = cv2.face.LBPHFaceRecognizer_create()

    # Train: latih model dengan train_faces dan train_labels, kemudian save model
    face_recognizer.train(train_faces, train_labels)
    face_recognizer.save(MODEL_PATH)

    # Test: load test data (test_faces & test_labels), validasikan juga
    test_faces, test_labels, _ = load_data(TEST_PATH, class_name)
    if test_faces is None:
        return
    
    # Test: inisialisasi correct_prediction & total
    correct_prediction = 0
    total = len(test_faces)

    # Test: gunakan for loop untuk uji setiap data test_faces
    for face_img, true_label in zip(test_faces, test_labels):
        predicted_label, confidence = face_recognizer.predict(face_img)

        # dapatkan true_name dan predicted_name
        true_name = class_name[true_label]
        predicted_name = class_name[predicted_label]

        # bandingkan kedua label
        if true_label == predicted_label:
            correct_prediction += 1
            Status = "CORRECT"
        else:
            Status = "WRONG"

        print(
            f"Prediction: {predicted_name} | "
            f"Actual: {true_name} | "
            f"Confidence: {confidence:.2f} | "
            f"Status: {Status}"
        )
    
    # Test: dapatkan avg_accuray
    avg_accuracy = (correct_prediction/total) * 100
    print(f"Average Accuracy: {avg_accuracy:.2f}")

In [21]:
def load_saved_model():
    if not os.path.exists(MODEL_PATH):
        print("WARNING: there is no model yet!")
        return None, None
    
    # inisialiasi face_recognizer dan masukkan MODEL_PATH ke dirinya
    face_recognizer = cv2.face.LBPHFaceRecognizer_create()
    face_recognizer.read(MODEL_PATH)
    class_name = sorted(os.listdir(TRAIN_PATH))
    return face_recognizer, class_name

In [22]:
def predict_image():
    # Input: ambil input path
    input_path = input("ENTER PATH")

    # Prediction: inisialisasi face_recognizer dan class_name
    face_recognizer, class_name = load_saved_model()
    if face_recognizer is None:
        return

    # Prediction: dapatkan img_gray, face_box, & face_img. Validasikan img_gray dan face_img
    gray_img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
    if gray_img is None:
        print("WARNING: could not read the input path")
        return
    
    face_img, face_box = detect_and_crop_face(gray_img)
    if face_img is None:
        print("WARNING: could not detect face")
        return
    
    # Prediction: lakukan prediksi, kemudian inisialisasi threshold, (x, y, w, h), dan predicted_name
    predicted_label, confidence = face_recognizer.predict(face_img)
    predicted_name = class_name[predicted_label]
    threshold = 70
    x, y, w, h = face_box

    # Prediction: cek jika confidence lebih kecil atau besar dari threhold
    if confidence < threshold:
        text = f"{predicted_name} | {confidence:.2f}"
        color = (0, 255, 0)
    else:
        text = "Unknown"
        color = (0, 0, 255)

    print(f"Prediction: {predicted_name}")
    print(f"Confidence: {confidence:.2f}")
    print(f"Face box location: x = {x}, y = {y}, w = {w}, h = {h}")

    # Plot: dapatkan display_img. cv2.rectagle, dan cv2.putText
    display_img = cv2.cvtColor(gray_img, cv2.COLOR_GRAY2BGR)
    cv2.rectangle(display_img, (x, y), (x+w, y+h), color, 2)
    cv2.putText(
        display_img,
        text,
        (x, max(y - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        color,
        2
    )

    # Plot: tayangkan display_img
    cv2.imshow("PREDIKSI", display_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [23]:
def menu():
    while True:
        print("\n1. Train\n2. Predict\n3. Exit")

        choice = input("ENTER CHOICE")

        if choice == "1":
            train_test_model()
        elif choice == "2":
            predict_image()
        elif choice == "3":
            print("Good Bye")
            break
        else:
            print("Invalid input")

In [24]:
menu()


1. Train
2. Predict
3. Exit
Training Model...
Loading: ./Dataset/train
Loading: ./Dataset/test
Prediction: s13 | Actual: s1 | Confidence: 85.71 | Status: WRONG
Prediction: s1 | Actual: s1 | Confidence: 80.74 | Status: CORRECT
Prediction: s1 | Actual: s1 | Confidence: 77.79 | Status: CORRECT
Prediction: s40 | Actual: s10 | Confidence: 67.97 | Status: WRONG
Prediction: s10 | Actual: s10 | Confidence: 52.23 | Status: CORRECT
Prediction: s11 | Actual: s11 | Confidence: 63.45 | Status: CORRECT
Prediction: s11 | Actual: s11 | Confidence: 53.36 | Status: CORRECT
Prediction: s11 | Actual: s11 | Confidence: 58.91 | Status: CORRECT
Prediction: s12 | Actual: s12 | Confidence: 60.61 | Status: CORRECT
Prediction: s12 | Actual: s12 | Confidence: 57.77 | Status: CORRECT
Prediction: s12 | Actual: s12 | Confidence: 52.20 | Status: CORRECT
Prediction: s13 | Actual: s13 | Confidence: 56.86 | Status: CORRECT
Prediction: s13 | Actual: s13 | Confidence: 50.94 | Status: CORRECT
Prediction: s13 | Actual: s13